In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [4]:
# Load the dataset
df = pd.read_csv(r'D:\Downloads\INTERNSHIP\House Price Prediction\House Price Prediction Dataset.csv')

# Display information about the dataset
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nStatistical Summary:")
print(df.describe())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (2000, 10)

First 5 rows:
   Id  Area  Bedrooms  Bathrooms  Floors  YearBuilt  Location  Condition  \
0   1  1360         5          4       3       1970  Downtown  Excellent   
1   2  4272         5          4       3       1958  Downtown  Excellent   
2   3  3592         2          2       3       1938  Downtown       Good   
3   4   966         4          2       2       1902  Suburban       Fair   
4   5  4926         1          4       2       1975  Downtown       Fair   

  Garage   Price  
0     No  149919  
1     No  424998  
2     No  266746  
3    Yes  244020  
4    Yes  636056  

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         2000 non-null   int64 
 1   Area       2000 non-null   int64 
 2   Bedrooms   2000 non-null   int64 
 3   Bathrooms  2000 non-null   int64 
 4   Floors     2000 non-null   

In [5]:
# Data Preprocessing

def preprocess_data(df):
    """
    Preprocess the house price dataset
    """
    # Make a copy to avoid modifying original
    data = df.copy()
    
    # Handling missing values
    numerical_cols = data.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        data[col].fillna(data[col].median(), inplace=True)
    
    # For categorical columns, fill with mode
    categorical_cols = data.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        data[col].fillna(data[col].mode()[0], inplace=True)
    
    # Encode categorical variables
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col].astype(str))
        label_encoders[col] = le
    
    # Remove outliers in numerical features
    # Using IQR method for square footage/area and bedrooms
    if 'area' in data.columns or 'sqft' in data.columns:
        area_col = 'area' if 'area' in data.columns else 'sqft'
        Q1 = data[area_col].quantile(0.25)
        Q3 = data[area_col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        data = data[(data[area_col] >= lower_bound) & (data[area_col] <= upper_bound)]
    
    # Feature engineering
    if 'area' in data.columns and 'price' in data.columns:
        data['price_per_sqft'] = data['price'] / data['area']
    
    return data, label_encoders

# Apply preprocessing
df_processed, encoders = preprocess_data(df)
print("\nProcessed Data Shape:", df_processed.shape)
print("\nProcessed Data Info:")
print(df_processed.info())


Processed Data Shape: (2000, 10)

Processed Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Id         2000 non-null   int64
 1   Area       2000 non-null   int64
 2   Bedrooms   2000 non-null   int64
 3   Bathrooms  2000 non-null   int64
 4   Floors     2000 non-null   int64
 5   YearBuilt  2000 non-null   int64
 6   Location   2000 non-null   int64
 7   Condition  2000 non-null   int64
 8   Garage     2000 non-null   int64
 9   Price      2000 non-null   int64
dtypes: int64(10)
memory usage: 156.4 KB
None


In [ ]:
# Feature selection and scaling

target_column = 'price'  # or 'Price' depending on your dataset

# Select features
feature_columns = [col for col in df_processed.columns if col != target_column 
                   and 'id' not in col.lower()]

X = df_processed[feature_columns]
y = df_processed[target_column]

print("Features:", feature_columns)
print("\nFeature Matrix Shape:", X.shape)
print("Target Vector Shape:", y.shape)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")